# LILY WAN 2.2 — P100/T4 FAST STUDIO
Deterministic stack: Torch 2.5.1 + CUDA 11.8 + ComfyUI v0.3.59 + Wan 2.2 TI2V 5B.
Run Cell 1. If it installs Torch and restarts, run Cell 1 again. When it says READY FOR CELL 2, run Cell 2.


In [ ]:
import os, subprocess, sys, time
def sh(a,check=True):
    p=subprocess.run(a,text=True,capture_output=True)
    print(p.stdout,end=''); print(p.stderr,end='')
    if check and p.returncode: raise RuntimeError('command failed: '+' '.join(a))
    return p
print('=== GPU PREFLIGHT ===')
s=sh(['nvidia-smi','--query-gpu=name,compute_cap,memory.total','--format=csv,noheader'],False)
first=s.stdout.splitlines()[0] if s.stdout.splitlines() else ''
if not first: raise RuntimeError('No NVIDIA GPU attached')
print(first)
def probe_and_version():
    try:
        import torch
        print('Torch:',torch.__version__,'CUDA:',torch.version.cuda,'GPU:',torch.cuda.get_device_name(0),'cap:',torch.cuda.get_device_capability(0))
        x=torch.ones((64,64),device='cuda',dtype=torch.float16); y=x@x; torch.cuda.synchronize()
        print('CUDA kernel probe: PASS',float(y[0,0]))
        return True, torch.__version__, str(torch.version.cuda)
    except Exception as e:
        print('CUDA kernel probe: FAIL',repr(e)); return False,'',''
ok,ver,cu=probe_and_version()
stack_ok=ok and ver.startswith('2.5.1+cu118') and cu=='11.8'
if not stack_ok:
    print('Installing the verified Kaggle GPU stack: Torch 2.5.1/cu118...')
    sh([sys.executable,'-m','pip','install','--no-cache-dir','--force-reinstall','torch==2.5.1','torchvision==0.20.1','torchaudio==2.5.1','--index-url','https://download.pytorch.org/whl/cu118'])
    print('Kernel restart required. RUN CELL 1 AGAIN after Kaggle reconnects.')
    time.sleep(2); os._exit(0)
print('VERIFIED STACK PASS')
print('READY FOR CELL 2')


In [ ]:
import os,sys,subprocess,time,json,secrets,shutil
from pathlib import Path
ROOT=Path('/kaggle/working/lily_wan_fast'); C=ROOT/'ComfyUI'; OUT=ROOT/'output'
ROOT.mkdir(parents=True,exist_ok=True); OUT.mkdir(parents=True,exist_ok=True)
def run(a,cwd=None,check=True):
    print('>', ' '.join(map(str,a)),flush=True)
    return subprocess.run(list(map(str,a)),cwd=cwd,check=check)
import torch
assert torch.cuda.is_available() and torch.__version__.startswith('2.5.1+cu118'), 'Run Cell 1 again first'
torch.cuda.synchronize()
print('[1/6] Installing the exact compatible runtime')
run([sys.executable,'-m','pip','install','--no-cache-dir','-q','gradio==5.50.0','huggingface_hub==0.36.0','safetensors==0.6.2','sentencepiece==0.2.1','einops==0.8.1','torchsde==0.2.6','av==15.1.0'])
# Never mix current ComfyUI with an old comfy-kitchen. Use the last verified pre-comfy-kitchen Wan 2.2 release instead.
if C.exists(): shutil.rmtree(C)
run(['git','clone','--depth','1','--branch','v0.3.59','https://github.com/Comfy-Org/ComfyUI.git',str(C)])
print('ComfyUI revision:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=C,text=True).strip())
req=[]
for q in (C/'requirements.txt').read_text().splitlines():
    q=q.strip(); low=q.lower()
    if q and not q.startswith('#') and not low.startswith(('torch','torchvision','torchaudio')): req.append(q)
if req: run([sys.executable,'-m','pip','install','--no-cache-dir','-q','--upgrade-strategy','only-if-needed',*req])
# Remove the incompatible package left by newer ComfyUI versions; v0.3.59 does not require it.
run([sys.executable,'-m','pip','uninstall','-y','comfy-kitchen'],check=False)
print('[2/6] Backend smoke test BEFORE downloading 17 GB of models')
PORT=8188
env=os.environ.copy(); env['PYTORCH_CUDA_ALLOC_CONF']='max_split_size_mb:128'
def start_backend(log_name):
    lf=open(ROOT/log_name,'w')
    p=subprocess.Popen([sys.executable,str(C/'main.py'),'--listen','127.0.0.1','--port',str(PORT),'--lowvram','--disable-xformers','--force-fp16','--fp32-vae','--reserve-vram','1.5'],cwd=C,env=env,stdout=lf,stderr=subprocess.STDOUT)
    return p,lf
import requests
smoke,smoke_log=start_backend('comfy_smoke.log')
started=False
for _ in range(90):
    try:
        if requests.get(f'http://127.0.0.1:{PORT}/system_stats',timeout=2).ok: started=True; break
    except Exception: pass
    if smoke.poll() is not None: break
    time.sleep(1)
if not started:
    smoke_log.flush(); smoke_log.close()
    raise RuntimeError('COMFYUI SMOKE TEST FAILED\n'+(ROOT/'comfy_smoke.log').read_text()[-12000:])
smoke.terminate()
try: smoke.wait(timeout=10)
except subprocess.TimeoutExpired: smoke.kill()
smoke_log.close(); time.sleep(2)
print('Backend smoke test: PASS')
print('[3/6] Downloading pinned official Wan 2.2 models')
from huggingface_hub import hf_hub_download
repo='Comfy-Org/Wan_2.2_ComfyUI_Repackaged'; rev='c4f60d30c55a624e35427060fdd217579a6c1d77'
files=[
 ('split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors','diffusion_models'),
 ('split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors','text_encoders'),
 ('split_files/vae/wan2.2_vae.safetensors','vae')
]
for remote,folder in files:
    d=C/'models'/folder; d.mkdir(parents=True,exist_ok=True); t=d/Path(remote).name
    cached=Path(hf_hub_download(repo_id=repo,filename=remote,revision=rev))
    if t.exists() or t.is_symlink(): t.unlink()
    t.symlink_to(cached)
    print('OK',t.name,round(cached.stat().st_size/1024**3,2),'GiB')
print('[4/6] Starting verified backend')
proc,log=start_backend('comfy.log')
for _ in range(120):
    try:
        if requests.get(f'http://127.0.0.1:{PORT}/system_stats',timeout=2).ok: break
    except Exception: pass
    if proc.poll() is not None:
        log.flush(); log.close(); raise RuntimeError((ROOT/'comfy.log').read_text()[-12000:])
    time.sleep(1)
else: raise RuntimeError('ComfyUI startup timeout')
print('[5/6] Verifying every workflow node against this exact ComfyUI release')
needed=['LoadImage','UNETLoader','CLIPLoader','VAELoader','CLIPTextEncode','Wan22ImageToVideoLatent','ModelSamplingSD3','KSampler','VAEDecodeTiled','SaveAnimatedWEBP']
all_info=requests.get(f'http://127.0.0.1:{PORT}/object_info',timeout=30).json()
missing=[n for n in needed if n not in all_info]
if missing: raise RuntimeError('Missing required ComfyUI nodes: '+', '.join(missing))
wan_info=all_info['Wan22ImageToVideoLatent']
if len(wan_info.get('output',[]))!=1 or wan_info.get('output',[None])[0]!='LATENT': raise RuntimeError('Unexpected Wan22ImageToVideoLatent schema: '+json.dumps(wan_info)[:1500])
print('Workflow schema check: PASS — Wan22ImageToVideoLatent has exactly one LATENT output')
print('[6/6] Launching mobile UI')
import gradio as gr
PRE={'⚡ TURBO':(320,192,33,8),'✨ NORMAL':(384,224,49,12),'👑 MAX':(448,256,65,18)}
def generate(img,prompt,preset,seed,progress=gr.Progress()):
    if img is None or not prompt.strip(): raise gr.Error('Upload an image and enter a prompt.')
    w,h,frames,steps=PRE[preset]; sid=int(seed)
    name=f'lily_{secrets.token_hex(6)}.png'; img.convert('RGB').save(C/'input'/name)
    wf={
      '1':{'class_type':'LoadImage','inputs':{'image':name}},
      '2':{'class_type':'UNETLoader','inputs':{'unet_name':'wan2.2_ti2v_5B_fp16.safetensors','weight_dtype':'default'}},
      '3':{'class_type':'CLIPLoader','inputs':{'clip_name':'umt5_xxl_fp8_e4m3fn_scaled.safetensors','type':'wan'}},
      '4':{'class_type':'VAELoader','inputs':{'vae_name':'wan2.2_vae.safetensors'}},
      '5':{'class_type':'CLIPTextEncode','inputs':{'text':prompt,'clip':['3',0]}},
      '6':{'class_type':'CLIPTextEncode','inputs':{'text':'blurry, frozen, distorted, watermark, text, low quality','clip':['3',0]}},
      '7':{'class_type':'Wan22ImageToVideoLatent','inputs':{'vae':['4',0],'width':w,'height':h,'length':frames,'batch_size':1,'start_image':['1',0]}},
      '8':{'class_type':'ModelSamplingSD3','inputs':{'model':['2',0],'shift':5.0}},
      '9':{'class_type':'KSampler','inputs':{'model':['8',0],'seed':sid,'steps':steps,'cfg':5.0,'sampler_name':'uni_pc','scheduler':'simple','positive':['5',0],'negative':['6',0],'latent_image':['7',0],'denoise':1.0}},
      '10':{'class_type':'VAEDecodeTiled','inputs':{'samples':['9',0],'vae':['4',0],'tile_size':256,'overlap':64,'temporal_size':16,'temporal_overlap':4}},
      '11':{'class_type':'SaveAnimatedWEBP','inputs':{'images':['10',0],'filename_prefix':'lily_wan','fps':12.0,'lossless':False,'quality':90,'method':'default'}}
    }
    r=requests.post(f'http://127.0.0.1:{PORT}/prompt',json={'prompt':wf},timeout=30)
    if not r.ok: raise gr.Error('Workflow rejected: '+r.text[:2200])
    pid=r.json()['prompt_id']; progress(.05,desc='Wan is generating…')
    while True:
        hist=requests.get(f'http://127.0.0.1:{PORT}/history/{pid}',timeout=10).json()
        if pid in hist:
            item=hist[pid]; imgs=item.get('outputs',{}).get('11',{}).get('images',[])
            if not imgs: raise gr.Error('Generation failed: '+json.dumps(item.get('status',{}))[:2200])
            f=imgs[0]
            data=requests.get(f'http://127.0.0.1:{PORT}/view',params={'filename':f['filename'],'subfolder':f.get('subfolder',''),'type':f.get('type','output')},timeout=60).content
            webp=OUT/f'lily_{sid}.webp'; webp.write_bytes(data)
            mp4=OUT/f'lily_{sid}.mp4'
            run(['ffmpeg','-y','-loglevel','error','-i',str(webp),'-vf','fps=24,scale=640:-2:flags=lanczos','-c:v','libx264','-preset','veryfast','-crf','20','-pix_fmt','yuv420p','-movflags','+faststart',str(mp4)])
            return str(mp4),str(mp4),f'DONE · {preset} · {frames} frames · {steps} steps · seed {sid}'
        time.sleep(2)
with gr.Blocks(title='Lily Wan 2.2 Fast Studio') as demo:
    gr.Markdown('# Lily Wan 2.2 Fast Studio')
    gr.Markdown('P100/T4-safe · Wan 2.2 TI2V 5B · start with ⚡ TURBO')
    image=gr.Image(type='pil',label='Start image')
    prompt=gr.Textbox(label='Motion prompt',lines=3)
    preset=gr.Radio(list(PRE),value='⚡ TURBO',label='Speed / quality')
    seed=gr.Number(value=42,precision=0,label='Seed')
    btn=gr.Button('GENERATE',variant='primary')
    video=gr.Video(label='Result'); download=gr.File(label='MP4'); status=gr.Textbox(label='Status')
    btn.click(generate,[image,prompt,preset,seed],[video,download,status],concurrency_limit=1)
demo.queue(max_size=2)
print('READY — open the Gradio share URL below')
demo.launch(share=True,server_name='0.0.0.0',prevent_thread_lock=True,allowed_paths=[str(OUT)])
